In [ ]:
import numpy as np
import torch
import torch.nn as nn

import h5py
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import tempfile
import os
import atexit
import glob
import shutil
from pathlib import Path

from dotenv import load_dotenv

from src.setup_model import setup_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

print("This can be run after preprocessing.")

In [ ]:
load_dotenv("config.env")

# --- Dataset parameters ---
USE_DVS = int(os.getenv("USE_DVS"))
USE_APS = int(os.getenv("USE_APS"))
MODEL_DIMS = USE_DVS + USE_APS
if MODEL_DIMS == 0: raise Exception("MODEL_DIMS must be non-zero")
PRETRAINED = bool(int(os.getenv("PRETRAINED")))
BINSIZE_MS = int(os.getenv("BINSIZE_MS"))
HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"
MODEL_TYPE = os.getenv("MODEL_TYPE")

BATCH_SIZE = int(os.getenv("BATCH_SIZE"))
WINDOW_SIZE = int(os.getenv("WINDOW_SIZE"))

# --- Location parameters ---
MODEL_SAVE_DIR = os.getenv("MODEL_SAVE_DIR")
DATA_DIR = os.getenv("PROCESSED_DIR")
OUT_DIR = os.getenv("OUT_DIR")
IMAGES_DIR = os.getenv("IMAGES_DIR")
TRAINING_DIR = f"{DATA_DIR}/Training"
TEST_DIR = f"{DATA_DIR}/Test"

In [ ]:
# SETUP MODEL
model = setup_model(MODEL_TYPE, MODEL_DIMS, PRETRAINED, device)

In [ ]:
model_files = glob.glob(MODEL_SAVE_DIR + "/*.pth")
best_model_dir = f"{MODEL_SAVE_DIR}/best/*.pth"
last_model_dir = f"{MODEL_SAVE_DIR}/last/*.pth"

model_files = sorted(model_files, key=os.path.getctime)

# pull best and last from a copy from tmpdir, in case they change while plotting (i.e. training is going on in parallel)
tmpdir = tempfile.mkdtemp()
atexit.register(lambda: shutil.rmtree(tmpdir, ignore_errors=True))
for path in glob.glob(best_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
for path in glob.glob(last_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
    # first best, then last
    model_files.extend(sorted(glob.glob(tmpdir + "/*.pth"), key=os.path.getctime))

train_files = sorted(glob.glob(TRAINING_DIR + "/*.hdf5"))
test_files = sorted(glob.glob(TEST_DIR + "/*.hdf5"))
whole_files = sorted(glob.glob(OUT_DIR + "/*.hdf5"))

In [ ]:
# Delete old images
try:
        os.mkdir(IMAGES_DIR)
except FileExistsError:
    for img in glob.glob(IMAGES_DIR + "/*"):
        os.remove(img)

In [ ]:
criterion = nn.MSELoss()
steering_slope = 1 / 180  # (output_max - output_min)/(input_max - input_min) = (1-(-1))/(180-(-180))
inv_steering_slope = 180

def generate_predictions(model, file_path, batch_size=BATCH_SIZE, window_size=WINDOW_SIZE):
    inv_apsdvs_max = 1 / 255
    def normalise(data):
            # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
            
            return (data * inv_apsdvs_max).astype(DTYPE)

    # used for getting the data out at the end.
    def denormalise_steering(data):
        return data * inv_steering_slope
    
    model.eval()

    with h5py.File(file_path, "r") as f:
        dvs_frame = np.array(f[DVS_FRAME_NAME])
        data_len = dvs_frame.shape[0]
        aps_frame = np.array(f[APS_FRAME_NAME])
        steering_angles = np.array(f["steering_wheel_angle"])
        if USE_APS and USE_DVS:
            # we have batch x 224 x 224 x 2 after stack
            combined_frames = np.stack((dvs_frame, aps_frame), axis=3)
        else:
            combined_frames = aps_frame if USE_APS else dvs_frame
            combined_frames = np.expand_dims(combined_frames, axis=3)
        
        preds, gts = [], []
        batched_window_size = batch_size * window_size
        for start in range(0, data_len, batched_window_size):
            end = min(start + batched_window_size, data_len)
            batch_data = combined_frames[start:end]

            # if the batch has fewer frames than a single window, skip processing
            if batch_data.shape[0] < window_size:
                continue
            # Only complete windows are processed; discard leftover frames
            num_windows = batch_data.shape[0] // window_size
            batch_data = batch_data[: num_windows * window_size]

            batch_data = normalise(batch_data)
            # we have batch_size * window_size, h, w, c
            # we want batch_size, window_size, c, h, w
            batch_data = batch_data.reshape(num_windows, window_size, *batch_data.shape[1:])
            # shuffle dimensions
            batch_data = batch_data.transpose(0, 1, 4, 2, 3)
            
            batch_data = torch.from_numpy(batch_data).to(device)
            if MODEL_TYPE == "Resnet":
                # Squeeze the sequence dimension
                b, t, c, w, h = batch_data.shape
                batch_data = batch_data.view(b * t, c, w, h)
            
            elif MODEL_TYPE != "ResnetRNN":
                batch_data = batch_data.permute(0, 2, 1, 3, 4)

            with torch.no_grad():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    batch_out = model(batch_data).cpu().numpy()
                    # if windowed and batched, output will be 3 dimensional like b, w, 1
                    # if not windowed, it will be 2 dimensional like b, 1
                    # we want to reshape to 1 dimension.
                    batch_out = batch_out.squeeze(-1).flatten()

                batch_out = denormalise_steering(batch_out)
                preds.append(batch_out)
                if model in ["Resnet", "ResnetRNN"]:
                    gts.append(steering_angles[start:start + num_windows * window_size])
                
                # only add last frame
                else:
                    gts.append(steering_angles[start: start + num_windows * window_size: window_size])
       
        preds = np.concatenate(preds, axis=0)
        gts = np.concatenate(gts, axis=0)

        # Convert to torch tensors
        preds_tensor = torch.tensor(preds).to(device)
        gts_tensor = torch.tensor(gts).to(device)

        # Calculate loss
        loss = criterion(preds_tensor, gts_tensor)

        return preds, loss

In [ ]:
def plot_predictions(model, files, model_files, dataset_name):
    fig = plt.figure(figsize=(15, 7.5 * len(files)))
    for i, f in enumerate(files):
        announce_str = f"Working on {f}, File [{i + 1}/{len(files)}]"
        print(f"{announce_str}\n{'-' * len(announce_str)}")
        ax = plt.subplot(len(files), 1, i + 1)
        ax.set_title(f, fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel("Predicted Values", fontsize=12)
        ax.grid(True)
        ax.set_ylim(-90, 90)
        ax.xaxis.set_major_locator(MultipleLocator(25))
        ax.tick_params(axis='x', rotation=45)
        with h5py.File(f, "r") as f5:
            gt = np.array(f5["steering_wheel_angle"])
        if len(gt) == 0:
            print("Empty dataset! Skipping.")
            continue
        time_axis = [x * (BINSIZE_MS / 1000) for x in range(len(gt))]
        ax.plot(time_axis, gt, label="truth", linewidth=2, linestyle='--')
        
        for model_file in model_files:
            epoch = Path(model_file).stem.split('_')[0]
            epoch = epoch.split('/')[-1][1:]
            loc = f"cuda:{0}"
            model_dict = torch.load(model_file, weights_only=True, map_location=loc)
            
            new_state_dict = {}
            for key, value in model_dict["model_state_dict"].items():
                new_state_dict[key.replace("module.", "")] = value
                
            model.load_state_dict(new_state_dict)
            model.to(device)

            preds, loss = generate_predictions(model, f)

            # if preds is shorter than gt (because e.g. one pred is made per 8 ground truths due to sequence size being 8) elongate preds.

            seq_len = len(gt) // len(preds)
            preds_stretched = np.repeat(preds, seq_len)
            # it's ok if it's still a bit short (< len(preds))


            # new time axis, it might be shorter since windows are considered.
            time_axis = [x * (BINSIZE_MS / 1000) for x in range(len(preds_stretched))]
            print(f"RMSE for model {epoch} is {loss ** 0.5:.2f}")
            ax.plot(time_axis, preds_stretched, label=f"E:{epoch}, L:{loss ** 0.5:.2f}")
            stretched_loss = criterion(
                torch.tensor(preds_stretched, dtype=torch.float32).to(device),
                torch.tensor(gt[:len(preds_stretched)], dtype=torch.float32).to(device)
            )
            print(f"RMSE for model {epoch} with stretched preds is {stretched_loss ** 0.5:.2f}")

        l = ax.legend(loc='best')
        if len(model_files) == 0:
            print("No models found! just plotting ground truth.")
        
        else:
            # add the "best" and "last" to legend
            # edge case, best and last are the same
            if len(l.get_texts()) > 2:
                current_text = l.get_texts()[-2].get_text()
                l.get_texts()[-2].set_text(f"Best: {current_text}")
                current_text = l.get_texts()[-1].get_text()
                l.get_texts()[-1].set_text(f"Last: {current_text}")
            else:
                current_text = l.get_texts()[-1].get_text()
                l.get_texts()[-1].set_text(f"Best/Last: {current_text}")

    plt.tight_layout()
    svg_path = os.path.join(IMAGES_DIR, f"{dataset_name}.svg")
    fig.savefig(svg_path, format="svg")
    print(f"Saved plot to {svg_path}")
    plt.show()
    plt.close(fig)

In [ ]:
# plot every 20th one to save time
plot_predictions(model, test_files[::20], model_files, "Test")

In [ ]:
# plot every 20th one to save time
plot_predictions(model, train_files[::20], model_files, "Train")

In [ ]:
plot_predictions(model, whole_files[:int(len(whole_files) / 3):1], model_files, "Unedited1of3")


In [ ]:
plot_predictions(model, whole_files[int(len(whole_files) / 3):2 * int(len(whole_files) / 3):1], model_files, "Unedited2of3")

In [ ]:
plot_predictions(model, whole_files[2 * int(len(whole_files) / 3)::1], model_files, "Unedited3of3")